In [4]:
from pathlib import Path
from kagglehub.config import DEFAULT_CACHE_FOLDER
import pandas as pd
import requests

CACHE_DIR = Path(DEFAULT_CACHE_FOLDER) / "competitions" / "hill-of-towie-wind-turbine-power-prediction"

def load_turbine_metadata(*, force_download: bool = False) -> pd.DataFrame:
    file_path = CACHE_DIR / "turbine_metadata.csv"
    if not file_path.exists() or force_download:
        response = requests.get(
            "https://zenodo.org/records/14870023/files/Hill_of_Towie_turbine_metadata.csv?download=1",
            headers={"Accept": "text/csv"},
            timeout=10,
        )
        response.raise_for_status()
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(response.content.decode("utf-8-sig"), encoding="utf-8")
    return pd.read_csv(file_path)

In [9]:
import pyproj
turbine_metadata = load_turbine_metadata()

# Define the coordinate reference systems
# WGS84 (lat/lon) - EPSG:4326
wgs84 = pyproj.CRS('EPSG:4326')

# British National Grid - EPSG:27700 (standard for UK/Scotland)
bng = pyproj.CRS('EPSG:27700')

# Create transformer object
transformer = pyproj.Transformer.from_crs(wgs84, bng, always_xy=True)

# Convert coordinates
easting, northing = transformer.transform(turbine_metadata["Latitude"].values, turbine_metadata["Longitude"].values)
turbine_metadata['Easting'] = easting
turbine_metadata['Northing'] = northing

In [14]:
easting_min, northing_min = turbine_metadata[["Easting", "Northing"]].min()

In [15]:
turbine_metadata["Easting"] = turbine_metadata["Easting"] - easting_min
turbine_metadata["Northing"] = turbine_metadata["Northing"] - northing_min

In [20]:
turbine_metadata["Northing"].values

array([  546.14317026,  1461.95686665,     0.        ,  1505.271253  ,
        2431.24223052,  3887.04174561,   619.72467653,  2440.977167  ,
        1270.23557988,  3926.92636852,  1614.88667628,  3156.73956947,
        2064.40939892,  2823.05550393,  5753.53353587,  7708.37782091,
        8888.35928206, 10376.4684385 , 10186.71494057, 10383.83870878,
       11680.77763174])

In [21]:
turbine_metadata

,Wind Farm,Turbine Name,Latitude,Longitude,Station ID,Manufacturer,Model,Rated power (kW),Hub Height (m),Rotor Diameter (m),Country,Commercial Operations Date,Easting,Northing
0,Hill of Towie,T01,57.499214,-3.086743,2304510,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,566.419250,546.143170
1,Hill of Towie,T02,57.496266,-3.082818,2304511,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,0.000000,1461.956867
2,Hill of Towie,T03,57.502070,-3.088980,2304512,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,1146.639831,0.000000
3,Hill of Towie,T04,57.501967,-3.082085,2304513,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,1264.295719,1505.271253
4,Hill of Towie,T05,57.498911,-3.078124,2304514,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,674.856257,2431.242231
5,Hill of Towie,T06,57.500245,-3.071322,2304515,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,1104.816160,3887.041746
6,Hill of Towie,T07,57.505133,-3.085851,2304516,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,1881.621064,619.724677
7,Hill of Towie,T08,57.504656,-3.077542,2304517,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,1945.770938,2440.977167
8,Hill of Towie,T09,57.508278,-3.082573,2304518,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,2637.622460,1270.235580
9,Hill of Towie,T10,57.505425,-3.070656,2304519,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12,2253.614427,3926.926369
